# Robust Deepfake Image Detection Under Real-World Post-Processing
### EfficientNet-B0 Walkthrough & Interactive Evaluation

This notebook provides an interactive walkthrough of the deepfake detection pipeline, demonstrating how real-world post-processing perturbations degrade standard models and how our robustness-enhanced EfficientNet-B0 model preserves high detection accuracy.

## 1. Setup & Environment

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt

from src.model import build_efficientnet_b0
from src.utils import (
    IMAGE_SIZE, MEAN, STD, CLASS_NAMES,
    jpeg_compression, resize_image, blur_image, add_noise, enhance_image
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Model Architecture: EfficientNet-B0

We adapt the EfficientNet-B0 backbone (5.3M parameters) by replacing the 1000-class ImageNet head with a dropout layer (p = 0.20) and a binary classification linear layer (1280 -> 2).

In [ ]:
model = build_efficientnet_b0(num_classes=2, pretrained=False)
print(model.classifier)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

## 3. Real-World Post-Processing Perturbations

Deepfakes uploaded to social media platforms undergo post-processing degradations that erase subtle synthesis artifacts. Here we visualize each perturbation implemented in our data pipeline.

In [ ]:
# Load a sample image
sample_img_path = PROJECT_ROOT / "results" / "demo_processed_gaussian_blur_r2.jpg"
if sample_img_path.exists():
    sample_img = Image.open(sample_img_path).convert("RGB")
else:
    import numpy as np
    sample_img = Image.fromarray((np.random.rand(256, 256, 3) * 255).astype("uint8"))

perturbations = {
    "Original": sample_img,
    "JPEG (Q=50)": jpeg_compression(sample_img, quality=50),
    "Resizing (112 -> 256)": resize_image(sample_img),
    "Gaussian Blur (r=2)": blur_image(sample_img, radius=2.0),
    "Gaussian Noise (s=0.05)": add_noise(sample_img, std=0.05, seed=42),
    "Enhancement": enhance_image(sample_img, contrast_factor=1.5, sharpness_factor=2.0)
}

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, (title, img) in zip(axes.flat, perturbations.items()):
    ax.imshow(img)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Empirical Benchmark Comparison (Table II)

Below are the empirical evaluation results comparing the Baseline and Robust EfficientNet-B0 models across all 6 test conditions.

In [ ]:
import pandas as pd

data = {
    "Condition": ["Clean", "JPEG Compression (Q=50)", "Resizing (112->256)", "Gaussian Blur (r=2.0)", "Gaussian Noise (s=0.05)", "Enhancement"],
    "Baseline Acc (%)": [98.67, 89.28, 93.68, 77.30, 68.32, 98.41],
    "Robust Acc (%)":   [97.05, 96.65, 96.63, 94.63, 93.07, 97.47],
    "Delta (%)":        [-1.62, +7.37, +2.95, +17.33, +24.75, -0.94]
}

df = pd.DataFrame(data)
print(df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(df))
ax.bar([i - 0.18 for i in x], df["Baseline Acc (%)"], width=0.36, label="Baseline EfficientNet-B0", color="#4C72B0")
ax.bar([i + 0.18 for i in x], df["Robust Acc (%)"], width=0.36, label="Robust EfficientNet-B0", color="#55A868")
ax.set_xticks(list(x))
ax.set_xticklabels(df["Condition"], rotation=25, ha="right")
ax.set_ylabel("Accuracy (%)")
ax.set_ylim(60, 105)
ax.set_title("Baseline vs Robust Model Accuracy Under Real-World Perturbations")
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()